In [2]:
import gymnasium as gym
import ale_py
import tqdm
import time

from tqdm.auto import tqdm
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
from huggingface_sb3 import package_to_hub


In [2]:
gym.register_envs(ale_py)

def env_make():

    env = gym.make("EnduroNoFrameskip-v4", render_mode = "rgb_array")
    env = AtariWrapper(env)
    return env


env = DummyVecEnv([env_make])

A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]


In [3]:
model = DQN(
    policy = "CnnPolicy",
    env = env,
    learning_rate = 1e-4,
    buffer_size = 100000,
    learning_starts = 10000,
    batch_size = 32,
    gamma = 0.99,
    train_freq = 4,
    gradient_steps = 1,
    target_update_interval = 1000,
    exploration_fraction = 0.2,
    exploration_final_eps = 0.01,
    optimize_memory_usage = False,
    verbose = 1
)

Using cpu device
Wrapping the env in a VecTransposeImage.


In [20]:
start = time.time()
model.learn(total_timesteps = 1e6)
end = time.time()
print(f"TOTAL TIME: {end-start}")

----------------------------------
| rollout/            |          |
|    exploration_rate | 0.934    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 597      |
|    time_elapsed     | 22       |
|    total_timesteps  | 13295    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.92e-07 |
|    n_updates        | 823      |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.868    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 406      |
|    time_elapsed     | 65       |
|    total_timesteps  | 26583    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 7.14e-08 |
|    n_updates        | 4145     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rat

In [21]:
model.save("EnduroNoFrameskip-v4")

In [8]:
video_env = VecVideoRecorder(
    env,
    video_folder = "results/",
    record_video_trigger = lambda step: step == 0,
    video_length = 1500,
    name_prefix = "Enduro"
)

In [23]:
obs = video_env.reset()

for _ in range(1500):
    action, states = model.predict(obs, deterministic = True)
    obs, rewards, dones, info = video_env.step(action)

video_env.close()

Saving video to /mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/results/Enduro-step-0-to-step-1500.mp4
MoviePy - Building video /mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/results/Enduro-step-0-to-step-1500.mp4.
MoviePy - Writing video /mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/results/Enduro-step-0-to-step-1500.mp4



MoviePy - Done !
MoviePy - video ready /mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/results/Enduro-step-0-to-step-1500.mp4


In [ ]:
path = "models/EnduroNoFrameskip-v4.zip"
model = DQN.load(path, env = env)
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
def make_env():
    env = gym.make("EnduroNoFrameskip-v4", render_mode = "rgb_array")
    env = AtariWrapper(env)
    return env

eval_env = DummyVecEnv([make_env])

package_to_hub(
    model = model,
    model_name = "Enduro",
    model_architecture = "DQN",
    env_id = "EnduroNoFrameskip-v4",
    eval_env = eval_env,
    repo_id = #"your repo id",
    commit_message = "Commited"
)



In [ ]:
#test
gym.register_envs(ale_py)

path = "models/EnduroNoFrameskip-v4.zip"

def make_env():
    env = gym.make("EnduroNoFrameskip-v4", render_mode = "human")
    env = AtariWrapper(env)
    return env

win_env = make_env()

model = DQN.load(path, env = win_env)

obs, info = win_env.reset()

while True:
    action, states = model.predict(obs, deterministic = True)
    obs, rewards, terminated, truncated ,info = win_env.step(action)

    if terminated or truncated:
        obs, info = win_env.reset()